<a href="https://colab.research.google.com/github/rirfan3689/CSCI323-Spam-Detection/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()  # Upload spam.csv

import pandas as pd
df = pd.read_csv('spam.csv', encoding='latin-1', sep='\t', header=None, names=['label', 'message'])
df = df.drop_duplicates()
print("Dataset loaded:", df.shape)

Saving spam.csv to spam.csv
Dataset loaded: (5169, 2)


In [2]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import numpy as np

print("Libraries loaded!")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Libraries loaded!


In [3]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize
    tokens = text.split()
    # Remove stopwords and stem
    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)

df['cleaned_message'] = df['message'].apply(preprocess_text)
print("Sample original:", df['message'][0])
print("Sample cleaned:", df['cleaned_message'][0])

Sample original: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Sample cleaned: go jurong point crazi avail bugi n great world la e buffet cine got amor wat


In [4]:
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['cleaned_message'])
y = df['label']

print("Feature matrix shape:", X.shape)
print("Labels shape:", y.shape)

Feature matrix shape: (5169, 5000)
Labels shape: (5169,)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)
print("\nTraining label distribution:")
print(pd.Series(y_train).value_counts())
print("\nTest label distribution:")
print(pd.Series(y_test).value_counts())

Training set size: (4135, 5000)
Test set size: (1034, 5000)

Training label distribution:
label
ham     3613
spam     522
Name: count, dtype: int64

Test label distribution:
label
ham     903
spam    131
Name: count, dtype: int64


In [6]:
import numpy as np
from scipy.sparse import save_npz

# Save sparse matrices and labels
save_npz('X_train.npz', X_train)
save_npz('X_test.npz', X_test)
np.save('y_train.npy', y_train)
np.save('y_test.npy', y_test)

print("Processed data saved successfully!")

Processed data saved successfully!
